# 📖 Notebook 4: Data Retention & Purging

Data retention answers one critical question: **"How long should we keep this data?"** Every day you store personal data is another day it could be breached, subpoenaed, or misused. In this notebook, we'll build automated retention policies that delete or anonymize data when its time is up.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to define retention policies per data type
- How to implement hard delete, soft delete, and anonymization purge strategies
- How to build an audit trail that proves compliance
- How to automate purging with a scheduled job
- Why audit logs are legally required (even after data is deleted)

## 🛠️ Setup

```bash
cd enterprise-patterns/privacy-review
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
from datetime import datetime, timedelta

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "privacy_review",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 📋 Step 1: Understand Our Retention Policies

Let's look at the retention policies we set up in our database. Each policy defines:
- **Which table** the policy applies to
- **How many days** data can be kept
- **What strategy** to use when the time is up (hard delete, soft delete, or anonymize)
- **Legal basis** — why this retention period was chosen

In [ ]:
def load_retention_policies():
    """Load all active retention policies from the database."""
    conn = get_db_connection()
    cursor = conn.cursor(psycopg2.extras.RealDictCursor)

    cursor.execute("""
        SELECT table_name, retention_days, description,
               legal_basis, purge_strategy, is_active
        FROM data_retention_policies
        WHERE is_active = TRUE
        ORDER BY retention_days
    """)

    policies = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return policies

policies = load_retention_policies()

print("📋 Active Data Retention Policies")
print("=" * 90)
print(f"  {'Table':<20} {'Days':>6} {'Strategy':<12} {'Legal Basis':<35} {'Description'}")
print("-" * 90)

strategy_icons = {"hard_delete": "🗑️", "soft_delete": "🏷️", "anonymize": "🔒"}
for p in policies:
    icon = strategy_icons.get(p["purge_strategy"], "?")
    days_label = str(p["retention_days"]) if p["retention_days"] > 0 else "immediate"
    print(f"  {p['table_name']:<20} {days_label:>6} {icon} {p['purge_strategy']:<10} "
          f"{p['legal_basis'][:34]:<35} {p['description']}")

print(f"\n💡 {len(policies)} active policies loaded")
print(f"   Strategies: 🗑️ hard_delete (remove rows), 🏷️ soft_delete (mark deleted), 🔒 anonymize (replace PII)")

## 🔍 Step 2: Find Data That Has Expired

Before we can purge anything, we need to identify which records have exceeded their retention period. We'll write a scanner that checks each table against its policy.

**Important**: We use the `created_at` timestamp to determine age. In a real system, you might use different timestamps for different tables (e.g., `resolved_at` for support tickets, `delivered_at` for orders).

In [ ]:
def find_expired_records(policy):
    """Find records that have exceeded their retention period."""
    table = policy["table_name"]
    retention_days = policy["retention_days"]

    if retention_days == 0:
        # Immediate deletion policies are handled differently
        return {"table": table, "expired_count": 0, "total_count": 0,
                "oldest_record": None, "note": "Immediate deletion — handled on user action"}

    cutoff_date = datetime.now() - timedelta(days=retention_days)

    conn = get_db_connection()
    cursor = conn.cursor()

    # Count expired records
    cursor.execute(
        f'SELECT COUNT(*) FROM "{table}" WHERE created_at < %s',
        (cutoff_date,)
    )
    expired_count = cursor.fetchone()[0]

    # Count total records
    cursor.execute(f'SELECT COUNT(*) FROM "{table}"')
    total_count = cursor.fetchone()[0]

    # Find oldest record
    cursor.execute(f'SELECT MIN(created_at) FROM "{table}"')
    oldest = cursor.fetchone()[0]

    conn.close()

    return {
        "table": table,
        "expired_count": expired_count,
        "total_count": total_count,
        "cutoff_date": cutoff_date,
        "oldest_record": oldest,
        "retention_days": retention_days,
        "strategy": policy["purge_strategy"]
    }

# Scan all tables
print("🔍 Retention Scan Results")
print("=" * 85)
print(f"  {'Table':<20} {'Total':>7} {'Expired':>8} {'%':>6} {'Strategy':<12} {'Cutoff Date'}")
print("-" * 85)

scan_results = []
for policy in policies:
    result = find_expired_records(policy)
    scan_results.append(result)

    if "note" in result:
        print(f"  {result['table']:<20} {'—':>7} {'—':>8} {'—':>6} {'—':<12} {result['note']}")
        continue

    pct = (result["expired_count"] / result["total_count"] * 100) if result["total_count"] > 0 else 0
    icon = "🔴" if pct > 50 else "🟡" if pct > 20 else "🟢"
    print(f"  {result['table']:<20} {result['total_count']:>7} {result['expired_count']:>8} "
          f"{icon}{pct:>5.1f}% {result['strategy']:<12} {result['cutoff_date'].strftime('%Y-%m-%d')}")

total_expired = sum(r["expired_count"] for r in scan_results if "expired_count" in r)
print(f"\n⚠️  Total records past retention: {total_expired}")
print(f"   These should be purged to comply with retention policies.")

## 🗑️ Step 3: Implement Purge Strategies

We have three purge strategies. Each handles expired data differently:

1. **Hard Delete** — remove the rows entirely (for data with no legal hold)
2. **Soft Delete** — mark as deleted but keep in DB (for grace periods)
3. **Anonymize** — replace PII with placeholder values but keep the record (for analytics)

All strategies must write to the **audit log** so we can prove to regulators what was deleted and why.

In [ ]:
def log_purge_action(cursor, table_name, records_affected, strategy, reason):
    """Write to the purge audit log. This is legally required."""
    cursor.execute("""
        INSERT INTO purge_audit_log
            (table_name, records_affected, purge_strategy, purge_reason, executed_by)
        VALUES (%s, %s, %s, %s, %s)
    """, (table_name, records_affected, strategy, reason, "retention-engine"))


def purge_hard_delete(table_name, retention_days):
    """Hard delete: remove rows older than retention period."""
    cutoff = datetime.now() - timedelta(days=retention_days)

    conn = get_db_connection()
    cursor = conn.cursor()

    # Count first (for audit log)
    cursor.execute(
        f'SELECT COUNT(*) FROM "{table_name}" WHERE created_at < %s',
        (cutoff,)
    )
    count = cursor.fetchone()[0]

    if count > 0:
        # Delete the records
        cursor.execute(
            f'DELETE FROM "{table_name}" WHERE created_at < %s',
            (cutoff,)
        )
        # Log the action
        log_purge_action(
            cursor, table_name, count, "hard_delete",
            f"Retention policy: {retention_days} days. Cutoff: {cutoff.isoformat()}"
        )

    conn.commit()
    conn.close()
    return count


def purge_anonymize(table_name, retention_days, pii_columns):
    """Anonymize: replace PII columns with placeholders but keep the row."""
    cutoff = datetime.now() - timedelta(days=retention_days)

    conn = get_db_connection()
    cursor = conn.cursor()

    # Count affected records
    cursor.execute(
        f'SELECT COUNT(*) FROM "{table_name}" WHERE created_at < %s',
        (cutoff,)
    )
    count = cursor.fetchone()[0]

    if count > 0:
        # Build SET clause to anonymize each PII column
        set_clauses = []
        for col, replacement in pii_columns.items():
            set_clauses.append(f'"{col}" = %s')

        set_sql = ", ".join(set_clauses)
        values = list(pii_columns.values()) + [cutoff]

        cursor.execute(
            f'UPDATE "{table_name}" SET {set_sql} WHERE created_at < %s',
            values
        )

        # Log the action
        log_purge_action(
            cursor, table_name, count, "anonymize",
            f"Anonymized columns: {list(pii_columns.keys())}. Cutoff: {cutoff.isoformat()}"
        )

    conn.commit()
    conn.close()
    return count


print("✅ Purge strategies loaded")
print("   - purge_hard_delete(): removes rows entirely")
print("   - purge_anonymize(): replaces PII with placeholders")

## ⚡ Step 4: Run the Purge Engine

Now let's execute the purges. We'll show before/after counts and what the audit log captures.

In [ ]:
# Before purge — snapshot current state
conn = get_db_connection()
cursor = conn.cursor()

tables = ["activity_log", "support_tickets", "orders"]
before_counts = {}
for table in tables:
    cursor.execute(f'SELECT COUNT(*) FROM "{table}"')
    before_counts[table] = cursor.fetchone()[0]

conn.close()

print("📊 Before Purge — Record Counts")
print("=" * 40)
for table, count in before_counts.items():
    print(f"  {table:<20} {count:>6} records")

print("\n🔄 Running purge engine...\n")

In [ ]:
# Execute purges according to each policy

purge_results = []

for policy in policies:
    table = policy["table_name"]
    strategy = policy["purge_strategy"]
    days = policy["retention_days"]

    if days == 0:
        continue  # Skip immediate-deletion policies

    if strategy == "hard_delete":
        count = purge_hard_delete(table, days)
        purge_results.append({"table": table, "strategy": strategy, "purged": count})

    elif strategy == "anonymize":
        # Define which columns to anonymize per table
        anonymize_columns = {
            "support_tickets": {
                "description": "[REDACTED — retention expired]",
                "internal_notes": "[REDACTED — retention expired]",
                "subject": "[Support ticket — anonymized]"
            },
            "orders": {
                "shipping_name": "[REDACTED]",
                "shipping_address": "[REDACTED]",
                "shipping_city": "[REDACTED]",
                "shipping_state": "[REDACTED]",
                "shipping_zip": "[REDACTED]"
            }
        }

        if table in anonymize_columns:
            count = purge_anonymize(table, days, anonymize_columns[table])
            purge_results.append({"table": table, "strategy": strategy, "purged": count})

# Display results
print("🗑️ Purge Results")
print("=" * 60)
print(f"  {'Table':<20} {'Strategy':<14} {'Records Affected':>18}")
print("-" * 60)

for r in purge_results:
    icon = "🗑️" if r["strategy"] == "hard_delete" else "🔒"
    print(f"  {r['table']:<20} {icon} {r['strategy']:<12} {r['purged']:>18}")

total_purged = sum(r["purged"] for r in purge_results)
print(f"\n  Total records processed: {total_purged}")

In [ ]:
# After purge — compare counts
conn = get_db_connection()
cursor = conn.cursor()

print("📊 After Purge — Record Counts")
print("=" * 55)
print(f"  {'Table':<20} {'Before':>8} {'After':>8} {'Change':>8}")
print("-" * 55)

for table in tables:
    cursor.execute(f'SELECT COUNT(*) FROM "{table}"')
    after = cursor.fetchone()[0]
    before = before_counts[table]
    change = after - before
    icon = "🗑️" if change < 0 else "🔒" if change == 0 else "?"
    print(f"  {table:<20} {before:>8} {after:>8} {icon} {change:>+7}")

conn.close()

print(f"\n💡 Note: 'orders' count stays the same because we anonymized (not deleted).")
print(f"   The rows exist but PII is replaced with '[REDACTED]'.")

## 🔍 Step 5: Verify Anonymization

Let's look at the anonymized records to confirm PII was actually removed.

In [ ]:
conn = get_db_connection()
cursor = conn.cursor(psycopg2.extras.RealDictCursor)

# Check anonymized orders
cursor.execute("""
    SELECT id, order_number, shipping_name, shipping_address,
           shipping_city, total, status, created_at
    FROM orders
    WHERE shipping_name = '[REDACTED]'
    LIMIT 5
""")

anonymized_orders = cursor.fetchall()

print("🔒 Anonymized Order Records")
print("=" * 90)

if anonymized_orders:
    for o in anonymized_orders:
        print(f"\n  Order {o['order_number']}:")
        print(f"    Shipping Name:    {o['shipping_name']}")
        print(f"    Shipping Address: {o['shipping_address']}")
        print(f"    Shipping City:    {o['shipping_city']}")
        print(f"    Total:            ${float(o['total']):.2f}")  # Financial data kept for tax
        print(f"    Status:           {o['status']}")
        print(f"    Created:          {o['created_at']}")

    print(f"\n💡 PII is gone, but financial totals and order metadata remain.")
    print(f"   This lets the tax team still use the data for compliance.")
else:
    print("  No anonymized orders found (orders may not have exceeded the 7-year retention).")
    print("  💡 In production, this would only trigger for very old records.")

# Check anonymized support tickets
cursor.execute("""
    SELECT id, subject, description, internal_notes, status, created_at
    FROM support_tickets
    WHERE description = '[REDACTED — retention expired]'
    LIMIT 5
""")

anonymized_tickets = cursor.fetchall()

print(f"\n🔒 Anonymized Support Tickets")
print("=" * 90)

if anonymized_tickets:
    for t in anonymized_tickets:
        print(f"\n  Ticket #{t['id']}:")
        print(f"    Subject:  {t['subject']}")
        print(f"    Content:  {t['description']}")
        print(f"    Notes:    {t['internal_notes']}")
        print(f"    Status:   {t['status']}")
else:
    print("  No anonymized tickets found (tickets may not have exceeded 1-year retention).")
    print("  💡 In production, tickets older than 365 days would be anonymized.")

conn.close()

## 📝 Step 6: The Audit Trail

The audit trail is **legally critical**. When a regulator asks "show me proof you deleted this user's data", you need to point to the audit log.

The audit log itself must **never** contain PII — it records *what* was deleted, *when*, *why*, and *how many records*, but NOT the actual data.

In [ ]:
def display_audit_trail():
    """Display the purge audit log."""
    conn = get_db_connection()
    cursor = conn.cursor(psycopg2.extras.RealDictCursor)

    cursor.execute("""
        SELECT id, table_name, records_affected, purge_strategy,
               purge_reason, executed_by, executed_at
        FROM purge_audit_log
        ORDER BY executed_at DESC
    """)

    logs = cursor.fetchall()
    conn.close()

    print("📝 Purge Audit Trail")
    print("=" * 100)

    if not logs:
        print("  No purge operations recorded yet.")
        return

    for log in logs:
        strategy_icon = "🗑️" if log["purge_strategy"] == "hard_delete" else "🔒"
        print(f"\n  {strategy_icon} Purge #{log['id']} — {log['executed_at']}")
        print(f"     Table:    {log['table_name']}")
        print(f"     Records:  {log['records_affected']}")
        print(f"     Strategy: {log['purge_strategy']}")
        print(f"     Reason:   {log['purge_reason']}")
        print(f"     By:       {log['executed_by']}")

    print(f"\n  📊 Total purge operations: {len(logs)}")
    print(f"  📊 Total records affected: {sum(l['records_affected'] for l in logs)}")

display_audit_trail()

## 🤖 Step 7: Build an Automated Retention Scheduler

In production, purging doesn't happen manually — it runs on a schedule (usually daily). Let's build a complete retention engine that could run as a cron job.

We'll also use Redis to track when the last purge ran, prevent duplicate runs, and cache policy lookups.

In [ ]:
class RetentionEngine:
    """Automated data retention and purging engine.
    
    This would run as a daily cron job in production:
    0 2 * * * python retention_engine.py  # Run at 2 AM daily
    """

    def __init__(self):
        self.redis = get_redis_client()
        self.run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

        # PII columns to anonymize per table
        self.anonymize_map = {
            "support_tickets": {
                "description": "[REDACTED — retention expired]",
                "internal_notes": "[REDACTED — retention expired]",
                "subject": "[Support ticket — anonymized]"
            },
            "orders": {
                "shipping_name": "[REDACTED]",
                "shipping_address": "[REDACTED]",
                "shipping_city": "[REDACTED]",
                "shipping_state": "[REDACTED]",
                "shipping_zip": "[REDACTED]"
            }
        }

    def acquire_lock(self):
        """Prevent multiple instances from running simultaneously."""
        # SET with NX (only if not exists) and EX (expire after 1 hour)
        acquired = self.redis.set(
            "retention:lock", self.run_id,
            nx=True, ex=3600
        )
        return acquired

    def release_lock(self):
        """Release the lock after purge completes."""
        self.redis.delete("retention:lock")

    def run(self):
        """Execute the full retention cycle."""
        print(f"🤖 Retention Engine — Run {self.run_id}")
        print("=" * 60)

        # Step 1: Acquire lock
        if not self.acquire_lock():
            print("❌ Another retention run is in progress. Exiting.")
            return
        print("🔒 Lock acquired")

        try:
            # Step 2: Load policies
            policies = load_retention_policies()
            print(f"📋 Loaded {len(policies)} policies")

            results = []

            # Step 3: Execute each policy
            for policy in policies:
                table = policy["table_name"]
                strategy = policy["purge_strategy"]
                days = policy["retention_days"]

                if days == 0:
                    continue

                print(f"\n  Processing: {table} (strategy={strategy}, retention={days} days)")

                if strategy == "hard_delete":
                    count = purge_hard_delete(table, days)
                elif strategy == "anonymize" and table in self.anonymize_map:
                    count = purge_anonymize(table, days, self.anonymize_map[table])
                else:
                    count = 0

                results.append({"table": table, "strategy": strategy, "purged": count})
                icon = "🗑️" if strategy == "hard_delete" else "🔒"
                print(f"  {icon} {count} records processed")

            # Step 4: Record run metadata in Redis
            run_summary = {
                "run_id": self.run_id,
                "completed_at": datetime.now().isoformat(),
                "policies_executed": len(results),
                "total_records_processed": sum(r["purged"] for r in results),
                "results": json.dumps(results)
            }

            self.redis.hset(f"retention:run:{self.run_id}", mapping=run_summary)
            self.redis.set("retention:last_run", self.run_id)
            # Keep run history for 90 days
            self.redis.expire(f"retention:run:{self.run_id}", 90 * 86400)

            # Step 5: Print summary
            print("\n" + "=" * 60)
            print("✅ Retention run complete")
            print(f"   Run ID: {self.run_id}")
            print(f"   Policies: {len(results)}")
            print(f"   Records: {run_summary['total_records_processed']}")

        finally:
            # Always release the lock
            self.release_lock()
            print("🔓 Lock released")

# Run the engine
engine = RetentionEngine()
engine.run()

In [ ]:
# Show the run history from Redis
r = get_redis_client()

last_run = r.get("retention:last_run")
if last_run:
    run_data = r.hgetall(f"retention:run:{last_run}")
    print("📊 Last Retention Run (from Redis)")
    print("=" * 50)
    for key, value in run_data.items():
        if key == "results":
            results = json.loads(value)
            print(f"  {key}:")
            for r_item in results:
                print(f"    - {r_item['table']}: {r_item['purged']} ({r_item['strategy']})")
        else:
            print(f"  {key}: {value}")

print(f"\n💡 In production, monitoring systems would alert if:")
print(f"   - A retention run fails or doesn't complete")
print(f"   - No run has executed in the past 48 hours")
print(f"   - An unusually large number of records were purged")

## 🎯 Key Takeaways

1. **Define retention before collecting data** — know how long you'll keep it before you start storing it
2. **Three strategies** — hard delete (gone forever), soft delete (grace period), anonymize (keep for analytics)
3. **Audit everything** — the audit log proves compliance to regulators
4. **Automate purging** — manual deletion doesn't scale and is error-prone
5. **Use locks** — prevent duplicate purge runs from corrupting data
6. **Monitor the engine** — alert if purging stops or behaves unexpectedly

### What Microsoft Does

- **Azure Data Lifecycle Management** automatically moves data through retention tiers
- **Microsoft 365 Retention Labels** let admins set retention per document
- **Legal Hold** can override retention policies during litigation
- **Data Subject Requests (DSRs)** must complete user deletion within 30 days
- Every Azure service must implement retention as part of its privacy review
- The compliance dashboard shows retention coverage across all services

### What You've Learned in This Series

| Notebook | Topic | Key Skill |
|----------|-------|-----------|
| 1 | Data Classification | Scan and classify PII in databases |
| 2 | Privacy Impact Assessment | Evaluate and score privacy risk before launch |
| 3 | Anonymization Techniques | Protect data with k-anonymity, DP, tokenization |
| 4 | Data Retention & Purging | Automate deletion with audit trails |

These four capabilities form the foundation of **privacy engineering** at any large tech company. Understanding them will help you build systems that respect user privacy and comply with regulations worldwide.